# Pandas Applying Functions (`apply()`)

## What is `apply()`?

`apply()` lets you run your own Python function on every value (or row) in a DataFrame.

Think of it as:

DataFrame → My Function → New Values

---

## Why use it?

Use `apply()` when Pandas doesn't already have a built-in function.

Common examples:

- Increase salaries by 5%
- Assign risk scores
- Categorize customers
- Create fraud flags
- Clean or transform data

---

## Example

```python
def projected_salary(salary):
    return salary * 1.05

df['salary_next_year'] = df['salary_year_avg'].apply(projected_salary)
```

### What's happening?

1. Create a function.
2. Pass the function to `.apply()`.
3. Pandas runs it for every salary.
4. Save the results into a new column.

---

## Remember

- **Function** = defines the logic.
- **`apply()`** = runs that logic on every value automatically.
- Similar to a `for` loop, but Pandas handles the looping for you.



In [22]:
# Importing Libraries
import pandas as pd                              # Pandas: work with and analyze tabular data
from datasets import load_dataset               # Hugging Face: load the job postings dataset
import matplotlib.pyplot as plt                 # Matplotlib: create charts and visualizations


# Loading Data
dataset = load_dataset('lukebarousse/data_jobs') # Download/load Luke's job dataset
df = dataset['train'].to_pandas()                # Convert the dataset into a Pandas DataFrame


# Data Cleanup
df['job_posted_date'] = pd.to_datetime(
    df['job_posted_date']
) 

In [23]:
# Select ONLY rows where 'salary_year_avg' is NOT missing (not NaN)
df[
    pd.notna(df['salary_year_avg'])      # Returns True for rows with a real salary
]['salary_year_avg']                     # Display only the salary_year_avg column

28        109500.0
77        140000.0
92        120000.0
100       228222.0
109        89000.0
            ...   
785624    139216.0
785641    150000.0
785648    221875.0
785682    157500.0
785692    157500.0
Name: salary_year_avg, Length: 22003, dtype: float64

In [24]:
help(df.apply)

Help on method apply in module pandas.core.frame:

apply(func: 'AggFuncType', axis: 'Axis' = 0, raw: 'bool' = False, result_type: "Literal['expand', 'reduce', 'broadcast'] | None" = None, args=(), by_row: "Literal[False, 'compat']" = 'compat', engine: "Callable | None | Literal['python', 'numba']" = None, engine_kwargs: 'dict[str, bool] | None' = None, **kwargs) method of pandas.DataFrame instance
    Apply a function along an axis of the DataFrame.
    
    Objects passed to the function are Series objects whose index is
    either the DataFrame's index (``axis=0``) or the DataFrame's columns
    (``axis=1``). By default (``result_type=None``), the final return type
    is inferred from the return type of the applied function. Otherwise,
    it depends on the `result_type` argument. The return type of the applied
    function is inferred based on the first computed result obtained after
    applying the function to a Series object.
    
    Parameters
    ----------
    func : functio

In [25]:
# Create a NEW independent DataFrame containing only rows with a valid salary.
# .copy() makes a separate copy so we can safely modify it
# (avoids Pandas' SettingWithCopyWarning).
df_salary = df[pd.notna(df['salary_year_avg'])].copy()

# Define a custom Python function
# Input: one salary value
# Output: salary increased by 3%
def projected_salary(salary):
    return salary * 1.03

# Create a NEW column called 'salary_year_inflated'
# Apply the function to every value in the salary_year_avg column
df_salary['salary_year_inflated'] = (
    df_salary['salary_year_avg'].apply(projected_salary)
)

# Display only the original salary and the new projected salary columns
df_salary[['salary_year_avg', 'salary_year_inflated']]

,salary_year_avg,salary_year_inflated
28,109500.0,112785.00
77,140000.0,144200.00
92,120000.0,123600.00
100,228222.0,235068.66
109,89000.0,91670.00
...,...,...
785624,139216.0,143392.48
785641,150000.0,154500.00
785648,221875.0,228531.25
785682,157500.0,162225.00


In [26]:
# Create (or overwrite) a new column with projected salaries
df_salary['salary_year_inflated'] = (

    # Select the salary column
    df_salary['salary_year_avg']

    # Apply a one-line (lambda) function to every salary
    .apply(
        lambda salary: salary * 1.03
        # lambda = create a quick one-line function
        # salary = the current salary value being processed
        # salary * 1.03 = increase the salary by 3%
    )
)

# Display only the original salary and the projected salary
df_salary[['salary_year_avg', 'salary_year_inflated']]

,salary_year_avg,salary_year_inflated
28,109500.0,112785.00
77,140000.0,144200.00
92,120000.0,123600.00
100,228222.0,235068.66
109,89000.0,91670.00
...,...,...
785624,139216.0,143392.48
785641,150000.0,154500.00
785648,221875.0,228531.25
785682,157500.0,162225.00


In [27]:
type(df['job_skills'][0])

float

In [28]:
# Import the ast module for converting list-like strings into real Python lists
import ast

# Find the first non-missing value in the job_skills column
first_skills_value = df['job_skills'].dropna().iloc[0]

# Convert the string representation into a real Python list
skills_list = ast.literal_eval(first_skills_value)

# Display the converted list
skills_list

['r', 'python', 'sql', 'nosql', 'power bi', 'tableau']

In [29]:
type(df['job_skills'][0 ])

float

In [30]:
df['job_skills'].head(10)

0                                                  NaN
1    ['r', 'python', 'sql', 'nosql', 'power bi', 't...
2    ['python', 'sql', 'c#', 'azure', 'airflow', 'd...
3    ['python', 'c++', 'java', 'matlab', 'aws', 'te...
4    ['bash', 'python', 'oracle', 'aws', 'ansible',...
5                             ['python', 'sql', 'gcp']
6    ['sql', 'python', 'java', 'sql server', 'gcp',...
7    ['sql', 'nosql', 'gcp', 'azure', 'aws', 'bigqu...
8                  ['excel', 'powerpoint', 'power bi']
9    ['sql', 'python', 'r', 'mongodb', 'mongodb', '...
Name: job_skills, dtype: str

In [31]:
df['job_skills'].first_valid_index()

1

In [32]:
type(df['job_skills'][18])

str

In [34]:

# Apply a function to every value in the 'job_skills' column.
df['job_skills'] = df['job_skills'].apply(

    # Lambda = a one-line anonymous function.
    lambda skill_list:

    # If the value is NOT missing (NaN), convert the string into a real Python list.
    ast.literal_eval(skill_list)

    # Otherwise, keep the missing value (NaN) unchanged.
    if pd.notna(skill_list) else skill_list
)

# Verify that the values are now Python lists (instead of strings).
type(df['job_skills'][0])


float

In [35]:
# Function to calculate the projected salary for each row
def projected_salary(row):

    # Check whether the job title contains the word "Senior"
    if "Senior" in row['job_title_short']:

        return 1.05 * row['salary_year_avg']      # Increase salary by 5%

    else:
        return row['salary_year_avg']             # Otherwise, keep the original salary

# Apply the function to every ROW (axis=1) and create a new column
df_salary['salary_year_inflated'] = df_salary.apply(projected_salary, axis=1)

# Display only these three columns
df_salary[['job_title_short', 'salary_year_avg', 'salary_year_inflated']]

,job_title_short,salary_year_avg,salary_year_inflated
28,Data Scientist,109500.0,109500.0
77,Data Engineer,140000.0,140000.0
92,Data Engineer,120000.0,120000.0
100,Data Scientist,228222.0,228222.0
109,Data Analyst,89000.0,89000.0
...,...,...,...
785624,Data Engineer,139216.0,139216.0
785641,Data Engineer,150000.0,150000.0
785648,Data Scientist,221875.0,221875.0
785682,Data Scientist,157500.0,157500.0
